<a href="https://colab.research.google.com/github/kamclar/brca-classification-tool/blob/main/notebooks/BRCA_ACMG_Criteria_Module1_v-1.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BRCA1/2 Variant Classification - Module 1: Automatic ACMG Criteria

This notebook is a first attempt to implement automatic evaluation of ACMG criteria for BRCA1 and BRCA2 variants.

We are following the ENIGMA VCEP v1.2 guidelines.. which are gene specific specifications built on top of the general ACMG/AMP 2015 framework.

The idea is.. for a given variant, we query public databases (gnomAD, ClinVar) and compute which criteria can be automatically assigned. Some criteria require manual review (like functional studies from literature) but many can be determined programmatically.

**References:**
- ENIGMA VCEP Specifications: https://cspec.genome.network/cspec/ui/svi/doc/GN092
- Original ACMG/AMP 2015: https://pubmed.ncbi.nlm.nih.gov/25741868/
- Parsons et al. 2024 (ENIGMA paper): https://www.cell.com/ajhg/fulltext/S0002-9297(24)00257-X

## Setup

We need requests for API calls.. thats basically it for now.

In [1]:
import requests
import json
import pandas as pd
import time
from typing import Optional, Dict, Any, List

## Test Variants

These are variants from a real classification exercise. They were used in EQA (External Quality Assessment) runs 13, 14, 15. Some are more straightforward, some are tricky edge cases. Viz excell file [add link]

The columns are:
- variant: HGVS notation (gene + c. notation + protein change)
- clinvar_class: what ClinVar says (can be conflicting)
- expert_class: what the expert classified it as (1-5 scale)
- type: missense, frameshift, nonsense, etc.

In [2]:
# variants from the classification exercise
# I am copying them manually from the excel file

test_variants = [
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.509G>A",
        "p_notation": "p.(Arg170Gln)",
        "variant_type": "missense",
        "clinvar_class": "conflict (4xVUS, 6xLB)",
        "expert_class": 1
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.1534C>T",
        "p_notation": "p.(Leu512Phe)",
        "variant_type": "missense",
        "clinvar_class": "1",
        "expert_class": 1
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.3668_3671dup",
        "p_notation": "p.(Cys1225fs)",
        "variant_type": "frameshift",
        "clinvar_class": "5",
        "expert_class": 5
    },
    {
        "gene": "BRCA2",
        "transcript": "NM_000059.4",
        "c_notation": "c.9097del",
        "p_notation": "p.(Thr3033fs)",
        "variant_type": "frameshift",
        "clinvar_class": "5",
        "expert_class": 5
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.5551_5552insT",
        "p_notation": "p.(Asp1851ValfsTer29)",
        "variant_type": "frameshift",
        "clinvar_class": "4",
        "expert_class": 4
    },
    {
        "gene": "BRCA2",
        "transcript": "NM_000059.4",
        "c_notation": "c.(793+1_794-1)_(1909+1_1910-1)del",
        "p_notation": "p.?",
        "variant_type": "exon_deletion",
        "clinvar_class": "NA",
        "expert_class": 3
    },
    {
        "gene": "BRCA2",
        "transcript": "NM_000059.4",
        "c_notation": "c.6147_6149del",
        "p_notation": "p.(Val2050del)",
        "variant_type": "inframe_deletion",
        "clinvar_class": "3",
        "expert_class": 2
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.3891_3893del",
        "p_notation": "p.(Ser1298del)",
        "variant_type": "inframe_deletion",
        "clinvar_class": "3",
        "expert_class": 3
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.4185G>A",
        "p_notation": "p.(Gln1395=)",
        "variant_type": "synonymous",
        "clinvar_class": "5",
        "expert_class": 5
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.628C>T",
        "p_notation": "p.(Gln210Ter)",
        "variant_type": "nonsense",
        "clinvar_class": "3",
        "expert_class": 3  # note: PTC in exon 10, NMD escape region
    },
    {
        "gene": "BRCA2",
        "transcript": "NM_000059.4",
        "c_notation": "c.8953+2T>C",
        "p_notation": "p.(?)",
        "variant_type": "splice_site",
        "clinvar_class": "4",
        "expert_class": 3
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.3247A>C",
        "p_notation": "p.(Met1083Leu)",
        "variant_type": "missense",
        "clinvar_class": "7xVUS, 1xLB",
        "expert_class": 2
    },
    {
        "gene": "BRCA1",
        "transcript": "NM_007294.4",
        "c_notation": "c.5217T>A",
        "p_notation": "p.(Asp1739Glu)",
        "variant_type": "missense",
        "clinvar_class": "",
        "expert_class": 4
    }
]

df_variants = pd.DataFrame(test_variants)
print(f"Loaded {len(df_variants)} test variants")
df_variants

Loaded 13 test variants


,gene,transcript,c_notation,p_notation,variant_type,clinvar_class,expert_class
0,BRCA1,NM_007294.4,c.509G>A,p.(Arg170Gln),missense,"conflict (4xVUS, 6xLB)",1
1,BRCA1,NM_007294.4,c.1534C>T,p.(Leu512Phe),missense,1,1
2,BRCA1,NM_007294.4,c.3668_3671dup,p.(Cys1225fs),frameshift,5,5
3,BRCA2,NM_000059.4,c.9097del,p.(Thr3033fs),frameshift,5,5
4,BRCA1,NM_007294.4,c.5551_5552insT,p.(Asp1851ValfsTer29),frameshift,4,4
5,BRCA2,NM_000059.4,c.(793+1_794-1)_(1909+1_1910-1)del,p.?,exon_deletion,NA,3
6,BRCA2,NM_000059.4,c.6147_6149del,p.(Val2050del),inframe_deletion,3,2
7,BRCA1,NM_007294.4,c.3891_3893del,p.(Ser1298del),inframe_deletion,3,3
8,BRCA1,NM_007294.4,c.4185G>A,p.(Gln1395=),synonymous,5,5
9,BRCA1,NM_007294.4,c.628C>T,p.(Gln210Ter),nonsense,3,3


## Functional Domains

ENIGMA defines specific functional domains for BRCA1 and BRCA2. These are regions where missense variants are more likely to be pathogenic.. because they affect important protein functions.

If a variant is OUTSIDE these domains AND has no predicted splicing effect.. it gets BP1_Strong (strong evidence towards benign). This is a key difference from general ACMG where BP1 is only supporting.

The domains are:
- BRCA1 RING domain (aa 2-101): E3 ubiquitin ligase activity
- BRCA1 coiled-coil (aa 1391-1424): protein-protein interaction
- BRCA1 BRCT repeats (aa 1650-1857): phosphopeptide binding
- BRCA2 PALB2 binding (aa 21-39)
- BRCA2 BRC repeats (aa 1002-2085): RAD51 binding
- BRCA2 DNA binding domain (aa 2481-3186)

Source: ENIGMA Specifications Appendix J

In [3]:
# functional domains as defined by ENIGMA
# these are amino acid positions

FUNCTIONAL_DOMAINS = {
    "BRCA1": {
        "RING": (2, 101),
        "coiled_coil": (1391, 1424),
        "BRCT": (1650, 1857)
    },
    "BRCA2": {
        "PALB2_binding": (21, 39),
        "BRC_repeats": (1002, 2085),
        "DBD": (2481, 3186)
    }
}

def get_amino_acid_position(p_notation: str) -> Optional[int]:
    """
    Extract the amino acid position from protein notation.
    Examples:
        p.(Arg170Gln) -> 170
        p.(Cys1225fs) -> 1225
        p.(Val2050del) -> 2050

    This is a bit crude but works for most cases..
    """
    import re
    # look for pattern like Arg170 or Cys1225
    match = re.search(r'[A-Z][a-z]{2}(\d+)', p_notation)
    if match:
        return int(match.group(1))
    return None

def is_in_functional_domain(gene: str, aa_position: int) -> tuple:
    """
    Check if an amino acid position is in a functional domain.
    Returns (is_in_domain, domain_name)
    """
    if gene not in FUNCTIONAL_DOMAINS:
        return (False, None)

    for domain_name, (start, end) in FUNCTIONAL_DOMAINS[gene].items():
        if start <= aa_position <= end:
            return (True, domain_name)

    return (False, None)

# test it
print("Testing domain lookup:")
print(f"  BRCA1 aa 50 -> {is_in_functional_domain('BRCA1', 50)}")
print(f"  BRCA1 aa 500 -> {is_in_functional_domain('BRCA1', 500)}")
print(f"  BRCA1 aa 1700 -> {is_in_functional_domain('BRCA1', 1700)}")
print(f"  BRCA2 aa 1500 -> {is_in_functional_domain('BRCA2', 1500)}")

Testing domain lookup:
  BRCA1 aa 50 -> (True, 'RING')
  BRCA1 aa 500 -> (False, None)
  BRCA1 aa 1700 -> (True, 'BRCT')
  BRCA2 aa 1500 -> (True, 'BRC_repeats')


## ClinVar Lookup

ClinVar is the main database for variant classifications. We can query it using the NCBI E-utilities API.

For our purposes we want to know:
- Is the variant already classified?
- What is the classification? (pathogenic, benign, VUS, conflicting)
- Is there an expert panel review? (ENIGMA submissions have higher weight)

We can also use ClinVar to find similar variants for PS1/PM5 criteria.. but thats more complex so we will do it later.

API documentation: https://www.ncbi.nlm.nih.gov/books/NBK25501/

In [4]:
# ClinVar lookup using NCBI E-utilities

NCBI_BASE_URL = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"

def search_clinvar(gene: str, c_notation: str) -> List[str]:
    """
    Search ClinVar for a variant.
    Returns list of ClinVar IDs (variation IDs).
    """
    # build search query
    # we search for gene name and the c. notation
    query = f"{gene}[gene] AND {c_notation}"

    url = f"{NCBI_BASE_URL}/esearch.fcgi"
    params = {
        "db": "clinvar",
        "term": query,
        "retmode": "json",
        "retmax": 10
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        id_list = data.get("esearchresult", {}).get("idlist", [])
        return id_list
    except Exception as e:
        print(f"Error searching ClinVar: {e}")
        return []

def fetch_clinvar_details(clinvar_id: str) -> Optional[Dict]:
    """
    Fetch details for a ClinVar variation ID.
    Returns dict with classification info.
    """
    url = f"{NCBI_BASE_URL}/esummary.fcgi"
    params = {
        "db": "clinvar",
        "id": clinvar_id,
        "retmode": "json"
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()

        result = data.get("result", {})
        if clinvar_id in result:
            return result[clinvar_id]
        return None
    except Exception as e:
        print(f"Error fetching ClinVar details: {e}")
        return None

def get_clinvar_classification(gene: str, c_notation: str) -> Dict:
    """
    Get ClinVar classification for a variant.
    Returns dict with:
        - found: bool
        - classification: str (pathogenic, likely_pathogenic, vus, likely_benign, benign, conflicting)
        - review_status: str
        - clinvar_id: str
    """
    ids = search_clinvar(gene, c_notation)

    if not ids:
        return {"found": False}

    # take the first result
    details = fetch_clinvar_details(ids[0])

    if not details:
        return {"found": False}

    # extract classification
    # ClinVar uses various field names..
    clinical_significance = details.get("clinical_significance", {}).get("description", "")
    review_status = details.get("clinical_significance", {}).get("review_status", "")

    return {
        "found": True,
        "classification": clinical_significance.lower(),
        "review_status": review_status,
        "clinvar_id": ids[0]
    }

# test it with one variant
print("Testing ClinVar lookup:")
result = get_clinvar_classification("BRCA1", "c.509G>A")
print(f"  BRCA1 c.509G>A -> {result}")
time.sleep(0.5)  # be nice to NCBI

Testing ClinVar lookup:
  BRCA1 c.509G>A -> {'found': True, 'classification': '', 'review_status': '', 'clinvar_id': '1763350'}


## gnomAD Lookup

gnomAD (Genome Aggregation Database) contains allele frequencies from large population studies. This is crucial for several ACMG criteria:

- **BA1** (Stand-alone Benign): FAF > 0.1% in non-founder population
- **BS1_Strong**: FAF > 0.01%
- **BS1_Supporting**: FAF > 0.002% and <= 0.01%
- **PM2_Supporting**: Variant absent from gnomAD (with read depth >= 25)

Important notes from ENIGMA:
- Use gnomAD v2.1 (exomes, non-cancer) AND v3.1 (genomes, non-cancer)
- Use FAF (filtering allele frequency) not raw AF
- Exclude founder populations (ASJ, FIN) from popmax
- **Do not apply PM2 for insertion/deletion/delins variants** (this is ENIGMA specific!)

gnomAD has a GraphQL API but its a bit complex.. for now I will use a simpler approach.

API: https://gnomad.broadinstitute.org/api

In [5]:
# gnomAD lookup
# this is simplified... real implementation would use the GraphQL API

GNOMAD_API_URL = "https://gnomad.broadinstitute.org/api"

def query_gnomad(variant_id: str) -> Optional[Dict]:
    """
    Query gnomAD for a variant.
    variant_id should be in format: chrom-pos-ref-alt (e.g. 17-43094464-G-A)

    Returns dict with allele frequencies or None if not found.

    Note: This is a placeholder.. the actual gnomAD API requires GraphQL queries
    which are more complex to set up.
    """
    # TODO: implement actual gnomAD API query
    # for now we return None and handle it in the criteria evaluation
    return None

def evaluate_frequency_criteria(gnomad_data: Optional[Dict], variant_type: str) -> Dict:
    """
    Evaluate frequency-based criteria (BA1, BS1, PM2).

    Returns dict with criteria that can be applied.
    """
    criteria = {}

    if gnomad_data is None:
        # variant not in gnomAD
        # check if we can apply PM2
        if variant_type in ["missense", "synonymous", "nonsense", "splice_site"]:
            # PM2 can be applied for these types (SNVs)
            criteria["PM2_Supporting"] = {
                "applies": True,
                "points": 1,
                "reason": "Absent from gnomAD (assuming read depth >= 25)"
            }
        else:
            # ENIGMA says: do not apply PM2 for indels
            criteria["PM2"] = {
                "applies": False,
                "points": 0,
                "reason": "PM2 not applicable for insertion/deletion/delins per ENIGMA guidelines"
            }
        return criteria

    # if we have gnomAD data, check frequencies
    faf = gnomad_data.get("faf", 0)  # filtering allele frequency

    if faf > 0.001:  # > 0.1%
        criteria["BA1"] = {
            "applies": True,
            "points": -99,  # stand-alone benign
            "reason": f"FAF {faf:.4f} > 0.001 (0.1%)"
        }
    elif faf > 0.0001:  # > 0.01%
        criteria["BS1_Strong"] = {
            "applies": True,
            "points": -4,
            "reason": f"FAF {faf:.4f} > 0.0001 (0.01%)"
        }
    elif faf > 0.00002:  # > 0.002%
        criteria["BS1_Supporting"] = {
            "applies": True,
            "points": -1,
            "reason": f"FAF {faf:.5f} > 0.00002 (0.002%)"
        }

    return criteria

# example usage
print("Example frequency criteria evaluation:")
print(f"  Absent SNV: {evaluate_frequency_criteria(None, 'missense')}")
print(f"  Absent indel: {evaluate_frequency_criteria(None, 'frameshift')}")
print(f"  FAF 0.0005: {evaluate_frequency_criteria({'faf': 0.0005}, 'missense')}")

Example frequency criteria evaluation:
  Absent SNV: {'PM2_Supporting': {'applies': True, 'points': 1, 'reason': 'Absent from gnomAD (assuming read depth >= 25)'}}
  Absent indel: {'PM2': {'applies': False, 'points': 0, 'reason': 'PM2 not applicable for insertion/deletion/delins per ENIGMA guidelines'}}
  FAF 0.0005: {'BS1_Strong': {'applies': True, 'points': -4, 'reason': 'FAF 0.0005 > 0.0001 (0.01%)'}}


## PVS1: Loss of Function Variants

PVS1 is for null variants.. nonsense, frameshift, canonical splice sites, initiation codon variants, and single/multi-exon deletions.

But its not that simple. ENIGMA has a detailed decision tree that considers:
- Which exon is affected
- Whether NMD (nonsense-mediated decay) is predicted
- Whether the variant is in the last exon (NMD escape)
- For BRCA1 exon 10 and BRCA2 exon 10: special handling (historically problematic region)

The strength of PVS1 can be:
- Very Strong (+8 points)
- Strong (+4 points)
- Moderate (+2 points)
- Supporting (+1 point)
- Not applicable

See ENIGMA Table 4 for full details: https://cspec.genome.network/cspec/ui/svi/doc/GN092

For now I will implement a simplified version..

In [6]:
# PVS1 evaluation - simplified version
# a proper implementation would need the full decision tree from ENIGMA Table 4

# BRCA1 has 23 exons, BRCA2 has 27 exons
# last exon = NMD escape region

GENE_INFO = {
    "BRCA1": {
        "last_exon": 23,
        "nmd_escape_exons": [23],  # last exon
        "special_exons": [10],  # historically problematic
        "total_aa": 1863
    },
    "BRCA2": {
        "last_exon": 27,
        "nmd_escape_exons": [27],
        "special_exons": [],
        "total_aa": 3418
    }
}

def evaluate_pvs1(gene: str, variant_type: str, p_notation: str) -> Dict:
    """
    Evaluate PVS1 criterion for loss-of-function variants.

    This is a simplified implementation. Real implementation would need:
    - Exact exon number where the variant occurs
    - Predicted termination codon position
    - NMD prediction
    - ENIGMA Table 4 lookup
    """
    result = {
        "applies": False,
        "strength": None,
        "points": 0,
        "reason": ""
    }

    # PVS1 only applies to certain variant types
    lof_types = ["frameshift", "nonsense", "splice_site", "exon_deletion"]
    if variant_type not in lof_types:
        result["reason"] = f"PVS1 not applicable for {variant_type} variants"
        return result

    # check for special cases
    aa_pos = get_amino_acid_position(p_notation)
    gene_info = GENE_INFO.get(gene, {})
    total_aa = gene_info.get("total_aa", 0)

    # very rough check: if the truncation is near the end, might be NMD escape
    # real implementation would check exact exon
    if aa_pos and total_aa:
        position_fraction = aa_pos / total_aa

        if position_fraction > 0.95:  # last 5% of protein
            # likely NMD escape region
            result["applies"] = True
            result["strength"] = "Moderate"
            result["points"] = 2
            result["reason"] = f"Truncation at aa {aa_pos} ({position_fraction:.1%} of protein) - possible NMD escape"
        else:
            # standard PVS1
            result["applies"] = True
            result["strength"] = "Very Strong"
            result["points"] = 8
            result["reason"] = f"Truncation at aa {aa_pos} ({position_fraction:.1%} of protein) - predicted NMD"
    else:
        # cant determine position, apply default
        if variant_type == "frameshift":
            result["applies"] = True
            result["strength"] = "Very Strong"
            result["points"] = 8
            result["reason"] = "Frameshift variant - predicted loss of function"
        elif variant_type == "nonsense":
            result["applies"] = True
            result["strength"] = "Very Strong"
            result["points"] = 8
            result["reason"] = "Nonsense variant - predicted loss of function"
        elif variant_type == "splice_site":
            # splice site variants need RNA analysis to confirm effect
            # without that, we might not apply PVS1 or apply at lower strength
            result["applies"] = False
            result["reason"] = "Splice site variant - requires RNA analysis (see ENIGMA PVS1 decision tree)"
        elif variant_type == "exon_deletion":
            # exon deletions depend on whether they are in-frame or out-of-frame
            result["applies"] = False
            result["reason"] = "Exon deletion - requires further analysis (in-frame vs out-of-frame)"

    return result

# test it
print("Testing PVS1 evaluation:")
print(f"  Frameshift at aa 1225: {evaluate_pvs1('BRCA1', 'frameshift', 'p.(Cys1225fs)')}")
print(f"  Nonsense at aa 210: {evaluate_pvs1('BRCA1', 'nonsense', 'p.(Gln210Ter)')}")
print(f"  Missense: {evaluate_pvs1('BRCA1', 'missense', 'p.(Arg170Gln)')}")

Testing PVS1 evaluation:
  Frameshift at aa 1225: {'applies': True, 'strength': 'Very Strong', 'points': 8, 'reason': 'Truncation at aa 1225 (65.8% of protein) - predicted NMD'}
  Nonsense at aa 210: {'applies': True, 'strength': 'Very Strong', 'points': 8, 'reason': 'Truncation at aa 210 (11.3% of protein) - predicted NMD'}
  Missense: {'applies': False, 'strength': None, 'points': 0, 'reason': 'PVS1 not applicable for missense variants'}


## BP1: Variants Outside Functional Domains

This is one of the criteria that ENIGMA upgraded from Supporting to Strong.

BP1_Strong applies when:
- Variant is missense, silent, or in-frame indel
- Variant is OUTSIDE a clinically important functional domain
- No predicted splicing effect (SpliceAI <= 0.1)

The logic is: if a variant doesnt affect a known functional region and doesnt affect splicing, its unlikely to be pathogenic.

For now we will check the domain part.. splicing prediction would require SpliceAI scores which we dont have yet.

In [7]:
def evaluate_bp1(gene: str, variant_type: str, p_notation: str, spliceai_score: float = 0) -> Dict:
    """
    Evaluate BP1 criterion (variant outside functional domain).

    BP1_Strong: silent/missense/in-frame variants outside functional domain
                AND no splicing prediction (SpliceAI <= 0.1)
    """
    result = {
        "applies": False,
        "strength": None,
        "points": 0,
        "reason": ""
    }

    # BP1 only applies to certain variant types
    applicable_types = ["missense", "synonymous", "inframe_deletion", "inframe_insertion"]
    if variant_type not in applicable_types:
        result["reason"] = f"BP1 not applicable for {variant_type} variants"
        return result

    # check splicing prediction
    if spliceai_score > 0.1:
        result["reason"] = f"SpliceAI score {spliceai_score} > 0.1 - possible splicing effect"
        return result

    # check if in functional domain
    aa_pos = get_amino_acid_position(p_notation)
    if aa_pos is None:
        result["reason"] = "Could not determine amino acid position"
        return result

    in_domain, domain_name = is_in_functional_domain(gene, aa_pos)

    if in_domain:
        result["reason"] = f"Variant at aa {aa_pos} is inside {domain_name} domain"
        return result

    # variant is outside functional domain and no splicing effect
    result["applies"] = True
    result["strength"] = "Strong"
    result["points"] = -4  # benign evidence
    result["reason"] = f"Variant at aa {aa_pos} is outside functional domains, no splicing predicted"

    return result

# test it
print("Testing BP1 evaluation:")
print(f"  Missense at aa 170 (outside domain): {evaluate_bp1('BRCA1', 'missense', 'p.(Arg170Gln)')}")
print(f"  Missense at aa 50 (in RING domain): {evaluate_bp1('BRCA1', 'missense', 'p.(Arg50Gln)')}")
print(f"  Missense at aa 1700 (in BRCT domain): {evaluate_bp1('BRCA1', 'missense', 'p.(Arg1700Gln)')}")

Testing BP1 evaluation:
  Missense at aa 170 (outside domain): {'applies': True, 'strength': 'Strong', 'points': -4, 'reason': 'Variant at aa 170 is outside functional domains, no splicing predicted'}
  Missense at aa 50 (in RING domain): {'applies': False, 'strength': None, 'points': 0, 'reason': 'Variant at aa 50 is inside RING domain'}
  Missense at aa 1700 (in BRCT domain): {'applies': False, 'strength': None, 'points': 0, 'reason': 'Variant at aa 1700 is inside BRCT domain'}


## PP3/BP4: In Silico Predictions

ENIGMA uses BayesDel for missense prediction and SpliceAI for splicing prediction.

**PP3 (evidence towards pathogenic):**
- Missense/in-frame in functional domain AND BayesDel >= 0.28
- OR SpliceAI >= 0.2 (any location)

**BP4 (evidence towards benign):**
- Missense/in-frame in functional domain AND BayesDel <= 0.15 AND SpliceAI <= 0.1
- Silent variant inside domain AND SpliceAI <= 0.1
- Intronic variant AND SpliceAI <= 0.1

Note: PP3 and BP4 are both Supporting strength only.

For this notebook we dont have access to BayesDel or SpliceAI scores.. so this is mostly placeholder code.

In [8]:
def evaluate_pp3_bp4(
    gene: str,
    variant_type: str,
    p_notation: str,
    bayesdel_score: Optional[float] = None,
    spliceai_score: Optional[float] = None
) -> Dict:
    """
    Evaluate PP3 and BP4 criteria based on in silico predictions.

    Returns dict with PP3 and/or BP4 evaluation.
    """
    results = {}

    # check if in functional domain
    aa_pos = get_amino_acid_position(p_notation)
    in_domain = False
    if aa_pos:
        in_domain, _ = is_in_functional_domain(gene, aa_pos)

    # PP3: splicing prediction (any variant type, any location)
    if spliceai_score is not None and spliceai_score >= 0.2:
        results["PP3"] = {
            "applies": True,
            "strength": "Supporting",
            "points": 1,
            "reason": f"SpliceAI score {spliceai_score} >= 0.2"
        }
    # PP3: missense prediction (only in functional domain)
    elif variant_type in ["missense", "inframe_deletion", "inframe_insertion"]:
        if in_domain and bayesdel_score is not None and bayesdel_score >= 0.28:
            results["PP3"] = {
                "applies": True,
                "strength": "Supporting",
                "points": 1,
                "reason": f"BayesDel score {bayesdel_score} >= 0.28 (in functional domain)"
            }

    # BP4: benign prediction
    if variant_type in ["missense", "inframe_deletion", "inframe_insertion"]:
        if in_domain:
            # need both BayesDel and SpliceAI to be low
            if (bayesdel_score is not None and bayesdel_score <= 0.15 and
                spliceai_score is not None and spliceai_score <= 0.1):
                results["BP4"] = {
                    "applies": True,
                    "strength": "Supporting",
                    "points": -1,
                    "reason": f"BayesDel {bayesdel_score} <= 0.15 AND SpliceAI {spliceai_score} <= 0.1"
                }
    elif variant_type == "synonymous":
        if in_domain and spliceai_score is not None and spliceai_score <= 0.1:
            results["BP4"] = {
                "applies": True,
                "strength": "Supporting",
                "points": -1,
                "reason": f"Silent variant in domain, SpliceAI {spliceai_score} <= 0.1"
            }

    if not results:
        results["PP3_BP4"] = {
            "applies": False,
            "reason": "No in silico scores available or criteria not met"
        }

    return results

# test it
print("Testing PP3/BP4 evaluation:")
print(f"  High SpliceAI: {evaluate_pp3_bp4('BRCA1', 'missense', 'p.(Arg170Gln)', spliceai_score=0.5)}")
print(f"  No scores: {evaluate_pp3_bp4('BRCA1', 'missense', 'p.(Arg170Gln)')}")

Testing PP3/BP4 evaluation:
  High SpliceAI: {'PP3': {'applies': True, 'strength': 'Supporting', 'points': 1, 'reason': 'SpliceAI score 0.5 >= 0.2'}}
  No scores: {'PP3_BP4': {'applies': False, 'reason': 'No in silico scores available or criteria not met'}}


## Putting It Together

Now lets run the evaluation on all our test variants and see what criteria we can automatically assign.

Keep in mind this is a simplified implementation.. we are missing:
- Actual gnomAD API queries
- BayesDel and SpliceAI scores
- Full PVS1 decision tree with exon-specific weights
- PM5 (similar pathogenic variants)
- PS1 (same amino acid change as known pathogenic)
- Literature-based criteria (PS3, PP1, PS4, etc.)

But it gives us a starting point..

In [9]:
def evaluate_variant(variant: Dict) -> Dict:
    """
    Evaluate all automatic criteria for a variant.
    Returns dict with all criteria evaluations and total score.
    """
    gene = variant["gene"]
    variant_type = variant["variant_type"]
    p_notation = variant["p_notation"]
    c_notation = variant["c_notation"]

    results = {
        "variant": f"{gene} {c_notation} {p_notation}",
        "expert_class": variant["expert_class"],
        "criteria": {},
        "total_points": 0
    }

    # PVS1
    pvs1 = evaluate_pvs1(gene, variant_type, p_notation)
    if pvs1["applies"]:
        results["criteria"]["PVS1"] = pvs1
        results["total_points"] += pvs1["points"]

    # BP1
    bp1 = evaluate_bp1(gene, variant_type, p_notation)
    if bp1["applies"]:
        results["criteria"]["BP1"] = bp1
        results["total_points"] += bp1["points"]

    # PM2 (simplified - we dont have gnomAD data)
    pm2 = evaluate_frequency_criteria(None, variant_type)
    for crit_name, crit_data in pm2.items():
        if crit_data.get("applies"):
            results["criteria"][crit_name] = crit_data
            results["total_points"] += crit_data["points"]

    # calculate predicted class based on points
    points = results["total_points"]
    if points >= 10:
        results["predicted_class"] = 5  # Pathogenic
    elif points >= 6:
        results["predicted_class"] = 4  # Likely Pathogenic
    elif points >= 0:
        results["predicted_class"] = 3  # VUS
    elif points >= -6:
        results["predicted_class"] = 2  # Likely Benign
    else:
        results["predicted_class"] = 1  # Benign

    return results

# run on all variants
print("Evaluating all test variants...")
print("=" * 80)

all_results = []
for variant in test_variants:
    result = evaluate_variant(variant)
    all_results.append(result)

    print(f"\n{result['variant']}")
    print(f"  Expert class: {result['expert_class']}")
    print(f"  Predicted class: {result['predicted_class']}")
    print(f"  Total points: {result['total_points']}")
    print(f"  Criteria applied:")
    for crit_name, crit_data in result["criteria"].items():
        print(f"    {crit_name}: {crit_data.get('points', 0)} points - {crit_data.get('reason', '')}")

Evaluating all test variants...

BRCA1 c.509G>A p.(Arg170Gln)
  Expert class: 1
  Predicted class: 2
  Total points: -3
  Criteria applied:
    BP1: -4 points - Variant at aa 170 is outside functional domains, no splicing predicted
    PM2_Supporting: 1 points - Absent from gnomAD (assuming read depth >= 25)

BRCA1 c.1534C>T p.(Leu512Phe)
  Expert class: 1
  Predicted class: 2
  Total points: -3
  Criteria applied:
    BP1: -4 points - Variant at aa 512 is outside functional domains, no splicing predicted
    PM2_Supporting: 1 points - Absent from gnomAD (assuming read depth >= 25)

BRCA1 c.3668_3671dup p.(Cys1225fs)
  Expert class: 5
  Predicted class: 4
  Total points: 8
  Criteria applied:
    PVS1: 8 points - Truncation at aa 1225 (65.8% of protein) - predicted NMD

BRCA2 c.9097del p.(Thr3033fs)
  Expert class: 5
  Predicted class: 4
  Total points: 8
  Criteria applied:
    PVS1: 8 points - Truncation at aa 3033 (88.7% of protein) - predicted NMD

BRCA1 c.5551_5552insT p.(Asp1851V

## Summary and Next Steps

This is a first pass at implementing automatic ACMG criteria for BRCA1/2 variants. As you can see, the results are incomplete because we are missing:

1. **gnomAD data** - we need actual API queries to get allele frequencies
2. **In silico scores** - BayesDel and SpliceAI are essential for PP3/BP4
3. **Full PVS1 decision tree** - exon-specific weights from ENIGMA Table 4
4. **ClinVar similar variants** - for PS1 and PM5
5. **Literature search** - for PS3, PP1, PS4 (this is Module 2)

The predicted classes will probably not match expert classifications until we have all these pieces.

### Questions to resolve:
- Do we want to use gnomAD API directly or download the data locally?
- How do we get BayesDel scores? (precomputed database or API)
- SpliceAI lookup options?

### Implementation priorities:
1. Get gnomAD working (BA1, BS1, PM2 are important)
2. Add SpliceAI lookup (affects many criteria)
3. Implement full PVS1 decision tree
4. Add ClinVar similar variant search

In [10]:
# summary table
summary_data = []
for r in all_results:
    summary_data.append({
        "variant": r["variant"],
        "expert": r["expert_class"],
        "predicted": r["predicted_class"],
        "points": r["total_points"],
        "criteria": ", ".join(r["criteria"].keys()) if r["criteria"] else "none",
        "match": "yes" if r["expert_class"] == r["predicted_class"] else "no"
    })

df_summary = pd.DataFrame(summary_data)
print("\nSummary:")
print(df_summary.to_string(index=False))

matches = sum(1 for r in all_results if r["expert_class"] == r["predicted_class"])
print(f"\nAccuracy: {matches}/{len(all_results)} ({100*matches/len(all_results):.0f}%)")
print("\n(Note: low accuracy expected - we are missing most data sources)")


Summary:
                                     variant  expert  predicted  points             criteria match
                BRCA1 c.509G>A p.(Arg170Gln)       1          2      -3  BP1, PM2_Supporting    no
               BRCA1 c.1534C>T p.(Leu512Phe)       1          2      -3  BP1, PM2_Supporting    no
          BRCA1 c.3668_3671dup p.(Cys1225fs)       5          4       8                 PVS1    no
               BRCA2 c.9097del p.(Thr3033fs)       5          4       8                 PVS1    no
 BRCA1 c.5551_5552insT p.(Asp1851ValfsTer29)       4          3       2                 PVS1    no
BRCA2 c.(793+1_794-1)_(1909+1_1910-1)del p.?       3          3       0                 none   yes
         BRCA2 c.6147_6149del p.(Val2050del)       2          3       0                 none    no
         BRCA1 c.3891_3893del p.(Ser1298del)       3          2      -4                  BP1    no
                BRCA1 c.4185G>A p.(Gln1395=)       5          3       1       PM2_Supporting    no
